# TaskFlow — SQL Fundamentals with Python and SQLite

## 1. Import SQLite and connect

`sqlite3` is included with Python. `sqlite3.connect(":memory:")` creates a temporary SQLite database held in memory for this connection. Use a filename such as `taskflow.db` when data should persist between program runs.


In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")

conn.execute("""
    CREATE TABLE cursor_demo (
        id INTEGER PRIMARY KEY,
        label TEXT NOT NULL
    )
""")

## 2. Connection and cursor

A **connection** represents access to a database. A **cursor** executes SQL and provides access to result sets.

`conn.execute(...)` is convenient for a one-off query: it executes the SQL and returns a cursor. Explicit cursors are useful when separate query result sets and their fetch positions need to be managed independently.

### Two cursors, one connection

Both cursors below use the same database connection but execute different queries. Fetching from one cursor does not consume the other cursor's results.


In [ ]:
conn.execute("""
    INSERT INTO cursor_demo (id, label)
    VALUES (1, 'First record')
""")

conn.execute("""
    INSERT INTO cursor_demo (id, label)
    VALUES (2, 'Second record')
""")

conn.commit()

In [ ]:
result = conn.execute("SELECT * FROM cursor_demo")

print(result.fetchall())

In [ ]:
conn.execute("""
    INSERT INTO cursor_demo (label)
    VALUES ('Third record')
""")

print(conn.execute("SELECT * FROM cursor_demo").fetchall())

In [ ]:
records = [
    (4, "Fourth record"),
    (5, "Fifth record")
]

conn.executemany("""
    INSERT INTO cursor_demo (id, label)
    VALUES (?, ?)
""", records)

conn.commit()

In [ ]:
result = conn.execute("SELECT * FROM cursor_demo")
print(result.fetchall())

In [ ]:
result = conn.execute("SELECT * FROM cursor_demo")
print(result.fetchone())

In [ ]:
print(result.fetchone())
print(result.fetchone())
print(result.fetchone())
print(result.fetchone())

In [ ]:
print(result.fetchone())

In [ ]:
result = conn.execute("SELECT * FROM cursor_demo")

print(result.fetchmany(2))

In [ ]:
print(result.fetchmany(2))

In [ ]:
result = conn.execute("SELECT * FROM cursor_demo")

print(result.fetchall())

In [ ]:
result = conn.execute("SELECT * FROM cursor_demo")

print("First row:", result.fetchone())
print("Next two rows:", result.fetchmany(2))
print("Remaining rows:", result.fetchall())

### `conn.execute()` versus `cursor.execute()`

Both approaches are valid. `conn.execute(...)` creates and returns a cursor for that operation, so an explicit cursor variable is not required for every query. Use the connection shortcut for concise one-off queries; use explicit cursors when you want to manage separate result sets or fetch positions.


In [ ]:
conn.execute("""
    CREATE TABLE customers (
        customer_id INTEGER PRIMARY KEY,
        name TEXT NOT NULL
    )
""")

conn.execute("""
    CREATE TABLE orders (
        order_id INTEGER PRIMARY KEY,
        customer_id INTEGER,
        amount REAL
    )
""")

conn.executemany(
    "INSERT INTO customers VALUES (?, ?)",
    [(1, "Asha"), (2, "Ravi")]
)

conn.executemany(
    "INSERT INTO orders VALUES (?, ?, ?)",
    [(101, 1, 500.0), (102, 2, 800.0), (103, 1, 300.0)]
)

conn.commit()

In [ ]:
orders_cursor = conn.cursor()
customers_cursor = conn.cursor()

orders_cursor.execute("SELECT * FROM orders")
customers_cursor.execute("SELECT * FROM customers")

print("First order:", orders_cursor.fetchone())
print("First customer:", customers_cursor.fetchone())

print("Next order:", orders_cursor.fetchone())
print("Next customer:", customers_cursor.fetchone())

In [ ]:
rows = conn.execute(
    "SELECT id, label FROM cursor_demo WHERE label LIKE ? ORDER BY id",
    ("%a%",)
).fetchall()
print(rows)


In [ ]:
rows = conn.execute("SELECT * FROM orders").fetchall()

for row in rows:
    print(row)

In [ ]:
rows

In [ ]:
for row in conn.execute("SELECT * FROM orders"):
    print(row)

## 3. SQL statement categories

These are common teaching categories; classifications can vary across SQL references.

| Category | Full form | Common statements | Purpose |
|---|---|---|---|
| DDL | Data Definition Language | `CREATE`, `ALTER`, `DROP` | Define database structures |
| DML | Data Manipulation Language | `INSERT`, `UPDATE`, `DELETE` | Change table records |
| DQL | Data Query Language | `SELECT` | Retrieve records |
| TCL | Transaction Control Language | `COMMIT`, `ROLLBACK` | Manage transactions |
| DCL | Data Control Language | `GRANT`, `REVOKE` | Manage permissions in systems that support them |

SQLite supports the DDL, DML, query, and transaction examples here. It does not implement standard `GRANT` and `REVOKE` statements like server databases such as PostgreSQL.


## 4. DDL — create the TaskFlow table

`CREATE TABLE` defines the table structure. A primary key identifies each task uniquely, and `NOT NULL` prevents required columns from being left as `NULL`.


In [34]:
conn.execute("""
    CREATE TABLE tasks2 (
        id INTEGER PRIMARY KEY,
        title TEXT NOT NULL,
        status TEXT NOT NULL,
        priority INTEGER NOT NULL,
        project_id INTEGER,
        assignee_id INTEGER,
        estimate_hours REAL
    )
""")

tables = conn.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name"
).fetchall()
print("Tables:", tables)


Tables: [('cursor_demo',), ('customers',), ('orders',), ('tasks',), ('tasks1',), ('tasks2',)]


## 5. DML — insert records

`INSERT` adds rows. `executemany()` runs one parameterized statement for multiple sets of values. The `?` placeholders keep values separate from SQL text. `commit()` commits the current transaction; it does not turn an in-memory database into a persistent file.


In [35]:
board = [
    (1, "Design schema", "done",        3, 10, 100, 4.0),
    (2, "Build API",     "in_progress", 5, 10, 101, 8.0),
    (3, "Write tests",   "todo",        4, 10, 100, 6.0),
    (4, "Fix login bug", "in_progress", 5, 11, 102, 3.0),
    (5, "Update docs",   "todo",        2, 11, 101, 2.0),
    (6, "Deploy v1",     "done",        4, 10, 102, 1.5),
]

conn.executemany("""
    INSERT INTO tasks2
        (id, title, status, priority, project_id, assignee_id, estimate_hours)
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", board)
conn.commit()

print("Tasks inserted:", conn.execute("SELECT COUNT(*) FROM tasks").fetchone()[0])


Tasks inserted: 6


## 6. DQL — select rows and columns

`SELECT` retrieves data. Selecting named columns makes the query's intent clearer than selecting every column with `*`.


In [ ]:
cur = conn.execute("""
    SELECT id, title, status
    FROM tasks
    ORDER BY id
""")
for row in cur.fetchall():
    print(row)


### Access result rows by column name

By default, rows are tuples. `sqlite3.Row` enables access by column name as well as position. Set `row_factory` before running queries whose results should use this row format.


In [ ]:
conn.row_factory = sqlite3.Row

cur = conn.execute("""
    SELECT id, title, status
    FROM tasks
    ORDER BY id
""")
for row in cur.fetchall():
    print(dict(row))


## 7. Filtering with `WHERE`

`WHERE` filters individual rows before grouping and aggregation. Parameterized values are passed separately from the SQL statement.


In [ ]:
cur = conn.execute("""
    SELECT id, title, status
    FROM tasks
    WHERE status IN (?, ?)
    ORDER BY id
""", ("todo", "in_progress"))

rows = [dict(row) for row in cur.fetchall()]
print("Matching tasks:", len(rows))
print("Task IDs:", [row["id"] for row in rows])
print(rows)


### Comparison operators and logical conditions

Common operators include `=`, `!=`, `>`, `<`, `>=`, and `<=`. `AND` requires both conditions to be true; `OR` requires at least one to be true. Use parentheses to make intended logic clear.


In [ ]:
cur = conn.execute("""
    SELECT id, title, priority, status
    FROM tasks
    WHERE status != 'done'
      AND priority >= 4
    ORDER BY priority DESC, id ASC
""")
for row in cur.fetchall():
    print(dict(row))


### `IN`, `BETWEEN`, `LIKE`, and `NULL`

- `IN` checks membership in a list of values.
- `BETWEEN` includes both endpoints.
- `LIKE` matches text patterns: `%` means zero or more characters; `_` means one character.
- Use `IS NULL` and `IS NOT NULL` to test missing values. `= NULL` is not the correct test.


In [ ]:
print("Priority 4 or 5:")
for row in conn.execute(
    "SELECT id, title, priority FROM tasks WHERE priority IN (?, ?) ORDER BY id",
    (4, 5)
):
    print(tuple(row))

print("\nEstimated hours between 2 and 4 inclusive:")
for row in conn.execute(
    "SELECT id, title, estimate_hours FROM tasks WHERE estimate_hours BETWEEN ? AND ? ORDER BY id",
    (2, 4)
):
    print(tuple(row))

print("\nTitles containing 'API':")
for row in conn.execute(
    "SELECT id, title FROM tasks WHERE title LIKE ? ORDER BY id",
    ("%API%",)
):
    print(tuple(row))

print("\nTasks with no assignee:")
for row in conn.execute(
    "SELECT id, title FROM tasks WHERE assignee_id IS NULL ORDER BY id"
):
    print(tuple(row))


## 8. Sorting, limiting, and distinct values

`ORDER BY` sorts results, `LIMIT` restricts the number of returned rows, and `OFFSET` skips rows. Use a deterministic tie-breaker such as a unique ID when predictable ordering matters. `DISTINCT` removes duplicate values from the query result.


In [ ]:
cur = conn.execute("""
    SELECT id, title, priority, estimate_hours
    FROM tasks
    WHERE status != 'done'
    ORDER BY priority DESC, estimate_hours DESC, id ASC
    LIMIT 3
""")
print("Top 3 non-done tasks:")
for row in cur.fetchall():
    print(row["id"], row["title"], row["priority"], row["estimate_hours"])

statuses = conn.execute(
    "SELECT DISTINCT status FROM tasks ORDER BY status"
).fetchall()
print("Distinct statuses:", [row["status"] for row in statuses])


## 9. Aggregate functions

`COUNT(*)` counts rows; `COUNT(column)` counts non-`NULL` values in that column. Most other common aggregates, such as `SUM()` and `AVG()`, ignore `NULL` inputs.


In [ ]:
cur = conn.execute("""
    SELECT COUNT(*) AS open_tasks,
           ROUND(SUM(estimate_hours), 2) AS total_hours,
           ROUND(AVG(priority), 2) AS avg_priority,
           MIN(priority) AS lowest_priority,
           MAX(priority) AS highest_priority
    FROM tasks
    WHERE status != 'done'
""")
print(dict(cur.fetchone()))


## 10. `GROUP BY` and `HAVING`

`GROUP BY` creates one group per distinct grouping value. Aggregate functions calculate a result for each group.

- `WHERE` filters rows before grouping.
- `HAVING` filters groups after aggregation.


In [ ]:
cur = conn.execute("""
    SELECT status,
           COUNT(*) AS task_count,
           ROUND(SUM(estimate_hours), 2) AS total_hours,
           ROUND(AVG(priority), 2) AS avg_priority
    FROM tasks
    GROUP BY status
    ORDER BY task_count DESC, status ASC
""")
for row in cur.fetchall():
    print(dict(row))

print("\nStatuses with more than one task:")
for row in conn.execute("""
    SELECT status, COUNT(*) AS task_count
    FROM tasks
    GROUP BY status
    HAVING COUNT(*) > 1
    ORDER BY status
"""):
    print(dict(row))


## 11. DML — update and delete

`UPDATE` changes existing rows, and `DELETE` removes rows. Always check the `WHERE` clause: without it, the statement can affect every row in the table.

These examples use a savepoint and roll back their changes so the original sample data remains available.


In [ ]:
conn.execute("SAVEPOINT demo_changes")

conn.execute("UPDATE tasks SET priority = ? WHERE id = ?", (1, 3))
print("Priority after update:", conn.execute(
    "SELECT priority FROM tasks WHERE id = 3"
).fetchone()["priority"])

conn.execute("DELETE FROM tasks WHERE id = ?", (5,))
print("Rows after delete:", conn.execute(
    "SELECT COUNT(*) FROM tasks"
).fetchone()[0])

conn.execute("ROLLBACK TO demo_changes")
conn.execute("RELEASE demo_changes")
print("Rows after rollback:", conn.execute(
    "SELECT COUNT(*) FROM tasks"
).fetchone()[0])


## 12. Joins — connect tasks, users, and projects

A relational application commonly stores users, projects, and tasks in separate tables. Joins combine related rows using key columns. This section uses a separate in-memory database to keep the example self-contained.


In [ ]:
db = sqlite3.connect(":memory:")
db.row_factory = sqlite3.Row
db.execute("PRAGMA foreign_keys = ON")

db.executescript("""
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);
CREATE TABLE projects (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL
);
CREATE TABLE tasks (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    status TEXT NOT NULL,
    priority INTEGER NOT NULL,
    estimate_hours REAL NOT NULL,
    project_id INTEGER REFERENCES projects(id),
    assignee_id INTEGER REFERENCES users(id)
);
""")

users = [(1, "Ana"), (2, "Ben"), (3, "Cara"), (4, "Dan")]
projects = [(10, "Apollo"), (20, "Beacon"), (30, "Comet")]
tasks = [
    (100, "Design schema", "done",        3, 4.0, 10, 1),
    (101, "Build API",     "in_progress", 5, 8.0, 10, 2),
    (102, "Write tests",   "todo",        2, 5.0, 10, 1),
    (103, "Setup CI",     "todo",        4, 3.0, 20, None),
    (104, "Deploy",       "review",       5, 2.0, 20, 3),
    (105, "Docs",         "done",         1, 6.0, 20, 2),
    (106, "Backfill logs","todo",         3, 4.0, 10, 3),
]

db.executemany("INSERT INTO users VALUES (?, ?)", users)
db.executemany("INSERT INTO projects VALUES (?, ?)", projects)
db.executemany("INSERT INTO tasks VALUES (?, ?, ?, ?, ?, ?, ?)", tasks)
db.commit()

print("Counts:", {
    "users": db.execute("SELECT COUNT(*) FROM users").fetchone()[0],
    "projects": db.execute("SELECT COUNT(*) FROM projects").fetchone()[0],
    "tasks": db.execute("SELECT COUNT(*) FROM tasks").fetchone()[0],
})


### `INNER JOIN`

An inner join returns rows that match the join conditions in both tables. Tasks without an assignee are excluded because there is no matching user row.


In [ ]:
for row in db.execute("""
    SELECT t.title,
           u.name AS assignee,
           p.name AS project
    FROM tasks AS t
    INNER JOIN users AS u ON u.id = t.assignee_id
    INNER JOIN projects AS p ON p.id = t.project_id
    WHERE t.status != 'done'
    ORDER BY p.name, t.title
"""):
    print(dict(row))


### `LEFT JOIN`

A left join keeps every row from the left table, even when there is no match on the right. Columns from the unmatched right-hand row appear as `NULL`.


In [ ]:
for row in db.execute("""
    SELECT t.title, u.name AS assignee
    FROM tasks AS t
    LEFT JOIN users AS u ON u.id = t.assignee_id
    ORDER BY t.id
"""):
    print(dict(row))


### Aggregate after a join

`COUNT(t.id)` counts matched tasks. For a project with no tasks, a `LEFT JOIN` still produces a row, but `t.id` is `NULL`; therefore `COUNT(t.id)` returns zero. `COUNT(*)` would count the unmatched join row.


In [ ]:
for row in db.execute("""
    SELECT p.name, COUNT(t.id) AS task_count
    FROM projects AS p
    LEFT JOIN tasks AS t ON t.project_id = p.id
    GROUP BY p.id, p.name
    HAVING COUNT(t.id) = 0
    ORDER BY p.name
"""):
    print(dict(row))


## 13. TCL — transactions, commit, and rollback

Transactions allow related changes to be committed together or rolled back if they should not be kept. This example commits setup work before explicitly beginning a new transaction.


In [ ]:
db.commit()
db.execute("BEGIN")

db.execute("UPDATE tasks SET priority = ? WHERE id = ?", (1, 101))
print("Temporary priority:", db.execute(
    "SELECT priority FROM tasks WHERE id = 101"
).fetchone()["priority"])

db.rollback()

print("Priority after rollback:", db.execute(
    "SELECT priority FROM tasks WHERE id = 101"
).fetchone()["priority"])


## 14. CRUD recap

CRUD means Create, Read, Update, and Delete. In SQL, these typically map to `INSERT`, `SELECT`, `UPDATE`, and `DELETE`. Use parameterized statements for values supplied by users or other application code.


In [ ]:
# Create
db.execute(
    "INSERT INTO tasks (id, title, status, priority, estimate_hours, project_id, assignee_id) "
    "VALUES (?, ?, ?, ?, ?, ?, ?)",
    (107, "Review release notes", "todo", 2, 1.5, 30, 4)
)
db.commit()

# Read
print(dict(db.execute(
    "SELECT id, title, status FROM tasks WHERE id = ?",
    (107,)
).fetchone()))

# Update
db.execute("UPDATE tasks SET status = ? WHERE id = ?", ("in_progress", 107))
db.commit()

# Delete
db.execute("DELETE FROM tasks WHERE id = ?", (107,))
db.commit()

print("Task 107 exists after delete:",
      db.execute("SELECT COUNT(*) FROM tasks WHERE id = 107").fetchone()[0])


## 15. Practice questions

Try writing the query before running the example solution.

1. Find all tasks that are not done.
2. Find tasks with priority 4 or 5.
3. Count tasks by status.
4. Calculate total estimated hours per project.
5. Find projects with no tasks.
6. Find tasks that have no assignee.
7. Return the three highest-priority tasks that are not done.


In [ ]:
# Example solution: find all tasks that are not done.
for row in db.execute(
    "SELECT id, title, status FROM tasks WHERE status != ? ORDER BY id",
    ("done",)
):
    print(dict(row))


## 16. Close connections

Close connections when they are no longer needed. Closing the final connection to an in-memory database destroys that database and its data.


In [ ]:
db.close()
conn.close()
print("Connections closed.")
